<a href="https://colab.research.google.com/github/samairaisabella-stack/Veritas-AI-Hackathon-2026/blob/main/Tvacha_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q gradio google-genai requests pillow

import gradio as gr
import requests
from PIL import Image
from google import genai
from google.colab import userdata

# Automatically pull key from Colab Secrets if saved as 'GEMINI_API_KEY'
try:
    default_key = userdata.get('GEMINI_API_KEY')
except Exception:
    default_key = ""

def analyze_skin_and_climate(api_key, city, image):
    active_key = api_key.strip() if api_key else default_key
    if not active_key:
        return "⚠️ Please paste your Gemini API Key (or save it in Colab Secrets as GEMINI_API_KEY)."
    if not city:
        return "⚠️ Please enter a city name."
    if image is None:
        return "⚠️ Please upload an image."

    try:
        # 1. Get City Coordinates
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"
        geo_res = requests.get(geo_url, timeout=10).json()

        if "results" not in geo_res or len(geo_res["results"]) == 0:
            return f"❌ Could not find city: {city}. Try a major city name."

        top_result = geo_res["results"][0]
        lat = top_result["latitude"]
        lon = top_result["longitude"]
        resolved_name = f"{top_result['name']}, {top_result.get('country', '')}"

        # 2. Get Live Weather & UV
        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m,relative_humidity_2m,uv_index"
        w_data = requests.get(weather_url, timeout=10).json()["current"]

        # 3. Get Live Air Pollution (PM2.5)
        air_url = f"https://air-quality-api.open-meteo.com/v1/air-quality?latitude={lat}&longitude={lon}&current=pm2_5"
        a_data = requests.get(air_url, timeout=10).json()["current"]

        climate_summary = (
            f"**Location:** {resolved_name} | "
            f"**Temp:** {w_data['temperature_2m']}°C | "
            f"**Humidity:** {w_data['relative_humidity_2m']}% | "
            f"**UV Index:** {w_data['uv_index']} | "
            f"**PM2.5:** {a_data['pm2_5']} µg/m³"
        )

        # 4. Call Gemini Vision AI
        client = genai.Client(api_key=active_key)
        prompt = f"""
        You are a Climate & Dermatological Health AI Assistant.
        Analyze the attached skin image alongside these real-time local climate conditions:
        {climate_summary}

        Provide a structured, concise response with:
        1. Visual Observation: What climate-related skin irritation might this resemble (e.g., heat rash, UV sunburn, pollution-triggered eczema, fungal irritation)?
        2. Climate Impact Analysis: Specifically explain how the current temperature, humidity, UV, or PM2.5 levels in {resolved_name} could worsen or trigger this condition.
        3. Actionable Protection Tips: 3 practical, non-prescription steps to protect the skin today.
        4. End with a bold medical disclaimer stating this is educational AI analysis, not a medical diagnosis.
        """

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[image, prompt]
        )

        return f"### 🌍 Live Local Climate Data\n{climate_summary}\n\n---\n\n### 🤖 AI Health & Climate Analysis\n{response.text}"

    except Exception as e:
        return f"❌ Error occurred: {str(e)}"

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🌡️ DermaClimate AI: Skin & Environment Risk Scanner")
    gr.Markdown("Upload a skin photo and enter your city to see how live heat, UV, and air pollution impact your skin health.")

    with gr.Row():
        with gr.Column():
            api_input = gr.Textbox(
                label="1. Gemini API Key",
                type="password",
                value=default_key,
                placeholder="Paste AIzaSy... key here (or set GEMINI_API_KEY in Colab Secrets)"
            )
            city_input = gr.Textbox(label="2. Your City", placeholder="e.g., Chandigarh, Delhi, London")
            img_input = gr.Image(type="pil", label="3. Upload Skin Photo")
            submit_btn = gr.Button("Analyze Skin & Climate Risk", variant="primary")

        with gr.Column():
            output_box = gr.Markdown(label="AI Analysis Result")

    submit_btn.click(
        fn=analyze_skin_and_climate,
        inputs=[api_input, city_input, img_input],
        outputs=output_box
    )

# debug=True prints live Python errors directly in your Colab cell output
demo.launch(share=True, debug=True)

/tmp/ipykernel_290/1275730293.py:77: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://346f1a5e125b773341.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://346f1a5e125b773341.gradio.live
